# Welcome!
This notebook demonstrates how to develop a conversational system that uses a deep knowledge base about hotels, combining structured instance-level data and an ontological model. The knowledge graph (KG) and ontology enable reasoning to enhance dialogue response generation. The task involves integrating a GraphRAG-like approach to query the knowledge base and generate accurate, context-aware responses.

Specifically, the notebook has the following steps:

1. **Setup**: Loading the knowledge graph, dialogues, and required libraries (e.g., OWLAPY).
2. **Analyzing the knowledge graph**: Exploring its structure and entities using OWLAPY queries.
3. **Extending the ontology**: Adding TBox information for expressive reasoning.
4. **Creating dialogues**: Create dialogues based on the examples. Write 5 simple dialogues and 5 more detailed ones to showcase different types of interactions.
5. **Combining ontology and KG data**: Deploying an OWL reasoner to perform class-expression queries.
6. **Query generation with LLMs**: Using an LLM (e.g., Llama3.2) to generate or assist in creating queries against the KG.
7. **Generating responses**: Summarizing retrieved data into dialogue responses using a KG-augmented RAG approach.
8. **Evaluation**: Assessing the system's performance using metrics like intersection-over-union scores.

## Assignment
The goal of this assignment is to develop a logic-enhanced conversational system that retrieves and reasons over domain knowledge to assist in dialogue response generation. You will focus on both the technical aspects of KG+ontology reasoning and the integration with LLMs for robust responses.

### Assignment Steps
1. **Analyze the provided knowledge graph and dialogues**:
   - Explore the KG's entities, properties, and relevance to the dialogues.
   - Identify opportunities where ontology reasoning enhances dialogue responses.
2. **Extend the ontology**:
   - Add expressive TBox information to support meaningful inferences.
3. **Deploy the reasoning environment**:
   - Use OWLAPY to combine the KG (as ABox) with the ontology for reasoning-based queries.
4. **Generate class-expression queries**:
   - Use instruction-based, few-shot prompting with Llama3.2 to produce or assist in creating the queries.
5. **Summarize results into dialogue responses**:
   - Apply KG-augmented RAG to generate user-facing answers based on reasoning results.
6. **Evaluate the system**:
   - Use appropriate metrics, including intersection-over-union scores for set-based answers.

## Report
Write a **5-page report** in LNCS format that includes:

1. **Introduction**: Background on conversational systems with LLMs and the role of reasoning over domain knowledge.
2. **Methodology**: A detailed description of your approach, including diagrams and examples.
3. **Results**: Evaluation findings from the implemented steps.
4. **Discussion**: Strengths and weaknesses of your approach, lessons learned, and potential improvements.

Make sure to use the following template: [Springer Lecture Notes in Computer Science](https://www.overleaf.com/latex/templates/springer-lecture-notes-in-computer-science/kzwwpvhwnvfj)


## Grading
Your work will be evaluated based on:

1. **Code Implementation (30%)**: Quality and functionality of the logic-enhanced conversational system.
2. **Report (70%)**: Depth of analysis and clarity in presenting methods, results, and lessons learned.

## Kaggle Environment Notes
To ensure smooth execution:
- Load the required data into `/kaggle/input/`.
- Use `/kaggle/working/` for saving temporary files.
- Turn on GPUs and internet connectivity when necessary, and follow best practices for resource management.

##### For colab: setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##### For colab pip installs + ollama installation

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama parsimonious rdflib SPARQLWrapper owlapy
# !pip install jpype1==1.5.2

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tgz
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 71.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Import libraries


##### All the imports + local graph init + ollama

In [1]:
import os
import re
import ollama # this is sufficient if local
import subprocess
import time
from owlready2 import *
from parsimonious.exceptions import IncompleteParseError
from rdflib import Graph, Literal, Namespace, URIRef, RDF, RDFS
from SPARQLWrapper import JSON, SPARQLWrapper

from owlapy import dl_to_owl_expression, manchester_to_owl_expression
from owlapy.class_expression import (
    OWLClass,
    OWLObjectHasValue,
    OWLObjectIntersectionOf,
    OWLObjectSomeValuesFrom
)
from owlapy.iri import IRI
from owlapy.owl_axiom import OWLObjectPropertyAssertionAxiom, OWLSubObjectPropertyOfAxiom
from owlapy.owl_individual import OWLNamedIndividual
from owlapy.owl_ontology import Ontology
from owlapy.owl_property import OWLObjectProperty
from owlapy.owl_reasoner import StructuralReasoner, SyncReasoner

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


In [95]:
from rdflib import RDF, RDFS, OWL

##### For colab setup

In [3]:
# for Colab RUN THIS WHEN USING OLLAMA IN COLAB (might need to run more than once,
# locally you need just pull llama3.2 once

#TODO: UNCOMMENT below for colab
subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull llama3.2 # get the llama model 3.2

### IMPORTANT: set to colab or local



In [2]:
# just use the extended_data relative path for local use
working_dir = "/content/drive/MyDrive/psai"
colab = True # HERE YOU CAN SET TO COLAB PATHS with True
if not colab:
    working_dir = ""
ttl_file_path = os.path.join(working_dir, "extended_data.ttl")
owl_file_path = os.path.join(working_dir, "extended_data.owl")

# might need to run this again if reasoning cannot find the graph
graph = Graph()
graph.parse(ttl_file_path, format="turtle")

<Graph identifier=Nb0102c46e7b3404cb70732ec35389233 (<class 'rdflib.graph.Graph'>)>

# ~~ 1. Analyze the provided knowledge graph (data.ttl).~~

### Don't run section 1 code, only output is needed here

In [ ]:
## the provided knowledge graph is in turtle (.ttl) format, which owlready2 has trouble
## parsing correctly in this environment. to avoid this issue, we first load the file
## using rdflib (which fully supports turtle), convert it to n-triples,
## and then load the converted graph into owlready2 for analysis.
## for the record owlready2 is an inner library used by owlapy.

from rdflib import Graph
from owlready2 import World

src = "/kaggle/input/w1-dataset3/extended_data.ttl"
dst = "/kaggle/working/data.nt"   ## .nt is the file format for n-triples

g = Graph()

g.parse(src, format="turtle")   ## here we parse the .ttl
g.serialize(destination=dst, format="nt")   # here its converted into .nt

world = World()
onto = world.get_ontology(f"file://{dst}").load(format="ntriples")

print("loaded into:", onto.base_iri)

In [ ]:
from collections import Counter
from rdflib import URIRef, Literal
import pandas as pd

# helper funcs
def is_uri(x):
    return isinstance(x, URIRef)

def is_lit(x):
    return isinstance(x, Literal)

def shorten(term, graph):
    # compact display using namespaces when possible

    if isinstance(term, URIRef):
        try:
            return term.n3(graph.namespace_manager)
        except Exception:
            return str(term)
    if isinstance(term, Literal):
        if term.language:
            return f"\"{str(term)[:60]}\"@{term.language}"
        if term.datatype:
            return f"\"{str(term)[:60]}\"^^{term.datatype}"
        return f"\"{str(term)[:60]}\""
    return str(term)


triples = list(g.triples((None, None, None)))
print("--- basic info ---")
print("triples:", len(triples))

subjects = set(s for s, p, o in triples)
predicates = set(p for s, p, o in triples)
objects = set(o for s, p, o in triples)

uris = set(x for x in subjects.union(objects) if is_uri(x))
lits = set(x for x in objects if is_lit(x))

print("unique subjects:", len(subjects))
print("unique predicates:", len(predicates))
print("unique objects:", len(objects))
print("unique URI nodes (subjects U objects):", len(uris))
print("unique literal nodes (objects):", len(lits))

pred_counts = Counter(p for s, p, o in triples)
top_preds = pred_counts.most_common(30)

print()
print("--- top predicates (by triple count) ---")

df_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "count"]
)
display(df_preds.head(30))

RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

type_triples = list(g.triples((None, RDF_TYPE, None)))
class_counts = Counter(o for s, p, o in type_triples if is_uri(o))

print()
print("--- types / classes (rdf:type) ---")
print("rdf:type triples:", len(type_triples))
print("distinct classes:", len(class_counts))

df_classes = pd.DataFrame(
    [(str(cls), shorten(cls, g), c) for cls, c in class_counts.most_common()],
    columns=["class_iri", "class", "instances_count"]
)
display(df_classes.head(30))

datatype_counts = Counter()
lang_counts = Counter()
lit_pred_counts = Counter()
lit_lengths = []

for s, p, o in triples:
    if is_lit(o):
        lit_pred_counts[p] += 1
        if o.datatype:
            datatype_counts[o.datatype] += 1
        else:
            datatype_counts[None] += 1
        if o.language:
            lang_counts[o.language] += 1
        lit_lengths.append(len(str(o)))

print()
print("--- literals ---")
print("literal objects:", sum(lit_pred_counts.values()))
print("predicates with literals:", len(lit_pred_counts))
print("avg literal length:", (sum(lit_lengths) / len(lit_lengths)) if lit_lengths else 0)

print("top literal predicates:")
for p, c in lit_pred_counts.most_common(20):
    print(f"{c:>7}  {shorten(p, g)}")

print("top datatypes:")
for dt, c in datatype_counts.most_common(15):
    dt_name = "no-datatype" if dt is None else shorten(dt, g)
    print(f"{c:>7}  {dt_name}")

print("top languages:")
for lang, c in lang_counts.most_common(10):
    print(f"{c:>7}  {lang}")

df_lit_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in lit_pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "literal_count"]
)
display(df_lit_preds.head(30))

--- basic info ---
triples: 9237
unique subjects: 1789
unique predicates: 14
unique objects: 870
unique URI nodes (subjects U objects): 1792
unique literal nodes (objects): 476

--- top predicates (by triple count) ---


,predicate_iri,predicate,count
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,rdf:type,3751
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:hasFacility,1640
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:location,1085
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:userRating,1000
4,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nationality,472
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:restaurantType,305
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:inCity,267
8,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:diet,137
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nextTo,34



--- types / classes (rdf:type) ---
rdf:type triples: 3751
distinct classes: 32


,class_iri,class,instances_count
0,http://www.w3.org/2002/07/owl#NamedIndividual,owl:NamedIndividual,1746
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Restaurant,506
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hotel,344
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Camping_Site,342
4,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hostel,314
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Neighbourhood,267
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Museum,55
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:City,34
8,http://www.w3.org/2002/07/owl#Class,owl:Class,33
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Trainstation,22



--- literals ---
literal objects: 485
predicates with literals: 1
avg literal length: 13.393814432989691
top literal predicates:
    485  rdfs:label
top datatypes:
    485  no-datatype
top languages:


,predicate_iri,predicate,literal_count
0,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485


# 2. Create a small ontology that can support expressive inference about hotels and analyse the dialogues (examples.txt).

##### Convert ttl to owl
(when making changes to ontology via ttl, use this to convert to owl for further usage in the notebook)

In [ ]:
# file_name = "extended_data" # Change to your filename (without extension)
# g = Graph().parse(f"{file_name}.ttl", format="turtle")
# g.serialize(destination=f"{file_name}.owl", format="xml")

##### Helper functions

In [4]:
def get_local_uri(label_text: str, file_path:str=owl_file_path) -> URIRef | None:
    """
    Converts wdt label to uri: eg Gulf to Q1322134.
    To ensure we can handle multiple name spaces. With owlapy manchester to owl, the issue
    is that it only takes one namespace as input. As we use ns1: and wdt: for our instances + classes
    + object properties this function is needed to replace with URIs
    """
    q = """
    SELECT ?s WHERE {
        ?s ?p ?label .
        FILTER(STR(?label) = "%s")
    } LIMIT 1
    """ % label_text

    results = graph.query(q)
    for row in results:
        return row.s
    return None

uri = get_local_uri("Gulf") # test
print(uri)

http://www.wikidata.org/entity/Q1322134


In [130]:
def resolve_manchester_fillers(query: str) -> str:
    """
    Takes as input a query, then uses previous function along with re lib
    to swap all wdt instances / classes in a query to the correct URIs (only for wdt).
    The output query can almost be used to convert to OWL class expression.
    """
    keywords = {'and', 'or', 'some', 'only', 'value', 'min', 'max', 'exactly', 'not', 'that'}

    def replacement_logic(match):
        word = match.group(0)

        if word.lower() in keywords:
            return word

        uri = get_local_uri(word)

        if uri:
            uri_str = str(uri)
            if "wikidata.org" in uri_str:
                return uri_str #uri_str.split('/')[-1]

        return word

    return re.sub(r'\b\w+\b', replacement_logic, query)
_NS1 = Namespace("http://kai.cs.vu.nl/2024/situated-minor-project/hotel#")

def parse_and_fix_manchester(query: str) -> str:
    pattern = r"(\w+)\s+(some|value)\s+([^\s()]+)"

    def replacement(match):
        prop = match.group(1)
        target = match.group(3).strip()
        # print(target)
        if target.startswith("http"):
            target_uri = URIRef(target)
        else:
            target_uri = _NS1[target]

        is_individual = (target_uri, RDF.type, OWL.NamedIndividual) in graph
        # print(target_uri, is_individual)
        new_op = "value" if is_individual else "some"
        return f"{prop} {new_op} {target}"

    return re.sub(pattern, replacement, query)
# test

# query = "Hotel and (inCountry value Italy) and (nextTo some BodyOfWater) and (inCity value Rome)"
# x = resolve_manchester_fillers(query)
# print(resolve_manchester_fillers(query))

In [6]:
NS1 = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#"

def get_iri(identifier: str) -> IRI:
    # Strip any remaining parentheses from the start or end of the string
    identifier = identifier.strip("()")
    if identifier.startswith("http"):
        return IRI.create(identifier)
    return IRI(NS1, identifier)

def parse_manchester_query(query_str: str) -> OWLObjectSomeValuesFrom | OWLObjectHasValue | OWLObjectIntersectionOf:
    """
    Parser that converts some manchester query to one of:
    OWLObjectSomeValuesFrom | OWLObjectHasValue | OWLObjectIntersectionOf.
    Handles n conjuncts >= 0.
    Lacks OR and NOT operators (TBD if will be added).
    """

    parts = re.split(r'\s+and\s+', query_str)
    expressions = []

    for part in parts:
        part = part.strip().strip("()")

        if " some " in part:
            prop_str, filler_str = part.split(" some ")
            prop = OWLObjectProperty(get_iri(prop_str))
            filler = OWLClass(get_iri(filler_str))
            expressions.append(OWLObjectSomeValuesFrom(property=prop, filler=filler))

        elif " value " in part:
            prop_str, filler_str = part.split(" value ")
            prop = OWLObjectProperty(get_iri(prop_str))
            filler = OWLNamedIndividual(get_iri(filler_str))
            # Fixed: parameter name is 'value', not 'individual'
            expressions.append(OWLObjectHasValue(property=prop, individual=filler))

        else:
            expressions.append(OWLClass(get_iri(part)))

    # If there is only one expression, return it directly instead of an IntersectionOf
    if len(expressions) == 1:
        return expressions[0]

    return OWLObjectIntersectionOf(expressions)

# Test with a single expression
query_single = "(inCountry value http://www.wikidata.org/entity/Q38) and Hotel"
complex_ce = parse_manchester_query(query_single)
print(f"Single Query Output: {complex_ce}")

# Test with a full conjunction
query_full = "(nextTo some BodyOfWater)"
complex_ce_full = parse_manchester_query(query_full)
print(f"Full Query Output: {complex_ce_full}")

Single Query Output: OWLObjectIntersectionOf((OWLObjectHasValue(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'inCountry')), individual=OWLNamedIndividual(IRI('http://www.wikidata.org/entity/', 'Q38'))), OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'Hotel'))))
Full Query Output: OWLObjectSomeValuesFrom(property=OWLObjectProperty(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'nextTo')),filler=OWLClass(IRI('http://kai.cs.vu.nl/2024/situated-minor-project/hotel#', 'BodyOfWater')))


In [125]:
# reason(
#        "Restaurant and (inCountry value Turkey) and (nextTo some Sea) and (hasFacility value Free_Wifi)",
#        "I want a hotel that has a restaurant with private bathrooms. Preferrably rated highly.")
y= "Restaurant and (inCountry some Turkey) and (hasFacility value Free_Wifi)"
# x = "Restaurant in Turkey in walking distance of a museum and with free wifi."
fix = resolve_manchester_fillers(y) # first convert all values to wdt uris if they are
fix1 = parse_and_fix_manchester(fix) # then convert to correct some and
print(fix)
print(fix1)
# fix2 = correct_query(x, fix1)


http://www.wikidata.org/entity/Q43
http://www.wikidata.org/entity/Q43 True
Free_Wifi
http://kai.cs.vu.nl/2024/situated-minor-project/hotel#Free_Wifi True
Restaurant and (inCountry some http://www.wikidata.org/entity/Q43) and (hasFacility value Free_Wifi)
Restaurant and (inCountry value http://www.wikidata.org/entity/Q43) and (hasFacility value Free_Wifi)


# Create your own dialogues

Once you have created your ontology, use it as the foundation for designing dialogues. Study the examples in examples.txt to understand their structure and content. Then, create 10 dialogues of your own, ensuring a range of difficulty levels: 5 simple ones and 5 more challenging ones. These dialogues should illustrate how your ontology can support reasoning and should include references to the types of information modeled in your ontology.

In [131]:
# Create 10 dialoges based on the description

dialogue1: str = "Find me an accomodation that is in Portugal and near a landmark." #exists in ABox
dialogue2: str = "I want to go to a camping by a river which is rated 5_stars."
dialogue3: str = "What are arabic restaurants that serve fast food."
dialogue4: str = "Find me all accomodations rated 5 stars."
dialogue5: str = "Hotels near the sea."
dialogue6: str = "I want a hotel with a sauna and a swimming pool, preferrably in Portugal near a train station." #for instance acco0
dialogue7: str = "Find me a hotel that is also a michelin restaurant in France. They should have free parking and a 24h front desk too."
dialogue8: str = "Can you find me a hostel in Porto with an airport shuttle. It would be nice if it is within walking distance from a museum."
dialogue9: str = "I want a French Restaurant in Turkey right by the sea with free wifi." #acco 36
dialogue10: str = "I want a hotel that has a restaurant with private bathrooms. Preferrably rated highly."
dialogues: list = [dialogue1, dialogue2, dialogue3, dialogue4, dialogue5, dialogue6, dialogue7, dialogue8, dialogue9, dialogue10]

# 3. Deploy a reasoning environment

Treated as the ABox in the OWL knowledge base. The idea is that instance queries
with complex class expressions should be used to retrieve different hotels, where
reasoning is crucial for many aspects. For example, a query "give me places that are
close to a coast" would also return places next to a beach if the query is evaluated
together with the ontology that states that a place next to the beach is next to a
coast.

In [8]:
namespace = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#"

# explicit cases + inferred cases
pellet_reasoner = SyncReasoner(
    ontology=owl_file_path,
    reasoner="Pellet"
    )

# Only explicit cases
# structural_reasoner = StructuralReasoner(
#     ontology=owl_file_path,
#     property_cache = True, negation_default = True, sub_properties = False
#     )

# 4. Instruct the LLM to produce the query or components of the query (e.g., keywords) against the KG

In [21]:
instruction : str = """### Role
You are a specialized NLP engine. Your sole task is to translate natural language user requests into formal Manchester Description Logic (DL) queries.

### Syntax Rules
1. **Format:** 'Class and (property some Class)' OR 'Class and (property value Individual)'.
2. **Naming:** Use PascalCase for Classes (e.g., Camping_Site) and camelCase for properties (e.g., inCountry).
3. **Operators:** and, or, not, some, value.
4. **Logic Selection:**
   - Use `value` for specific names (Italy, Vegan, Traditional) or specific facilities.
   - Use `some` for general categories (Sea, Restaurant, BodyOfWater).
5. **Nesting:** If a user mentions a feature within a place (e.g., a hotel with a gym), nest it: `Hotel and (hasFacility value Gym)`.

### Vocabulary Reference
- **Properties:** hasFacility, location, userRating, nationality, restaurantType, inCity, diet, nextTo, inCountry, walkingDistance.
- **Classes:** Restaurant, Hotel, Camping_Site, Hostel, Neighbourhood, Museum, City, TrainStation, PublicTransport, BodyOfWater, TouristAttraction, RestaurantType, UserRating, Country, Diet, Accommodation, River,
Gold River, Ocean, Lagoon, Drainage Basin, Bay, Gulf, Sea, Adjacent Sea, Mediterranean Sea,
Watercourse, Canal, Main Stream.
- **Facility Values:** Sauna, 24h_front_desk, Airport_Shuttle, Free_Wifi, Parking_Space, Private_Bathroom, RestaurantInHotel, Swimming_Pool.
- **RestaurantType Values:** FastFood, Fusion, Michelin, StreetFood, Traditional.
- **UserRating Values:** 1_stars, 2_stars, 3_stars, 4_stars, 5_stars.

### Translation Logic
- "Near" or "By" -> `nextTo`
- "In [Country]" -> `inCountry value [Country]`
- "Serving [Diet] food" -> `diet value [Diet]`
- "rating of atleast 4 stars" or "rated highly" -> `userRating some HighRating`
- "within walking distance of [Instance]" -> `walkingDistance value [Instance]`

### Output Requirement
- Output ONLY the query string.
- No explanations, no introductory text.
- No full stop.

### Examples
- User: "Hotel in Paris with a pool and a sauna."
- Query: Hotel and (inCity value Paris) and (hasFacility value Swimming_Pool) and (hasFacility value Sauna)

- User: "A restaurant next to a museum serving vegan food."
- Query: Restaurant and (nextTo some Museum) and (diet value Vegan)

- User: "Camping near a gold river in France close by some public transport."
- Query: "Camping_Site" and (nextTo some Gold River) and (inCountry value France) and (nextTo some PublicTransport)

- User: "Hotel with a high rating."
- Query: Hotel and (userRating some HighRating)

- User: "Hostel in Germany next to a river."
- Query: Hostel and (inCountry value Germany) and (nextTo some River)

- User: "I want to find chinese restaurants that serve vegan food."
- Query: Restautant and (nationality value Chinese) and (diet value Vegan)

- User: "Hotel rated 5 stars and within walking distance of a train station."
- Query: Hotel and (userRating value 5_stars) and (walkingDistance some TrainStation)
"""


In [30]:
#Step 2: Write a function that takes the model, instruction and one user question as input, runs the LLM and outputs its response
def question_to_query(instruction: str, question: str, model="llama3.2") -> str:
    '''
    This function is meant to use the instruction defined above to run the LLM in order to convert one user input
    question into a query for the ontology reasoner.
    Parameters: instruction (string), question (string), model version (string)
    Returns: LLM response (string)
    '''
    # get initial response to generate manchester syntax query
    response : str = ollama.chat(
        model=model,
        messages=[
            {'role': 'system', 'content': instruction},
            {'role': 'user', 'content': question},
        ],
        options={
            'temperature': 0
        }
    )
    manchester_syntax_query = response['message']['content']
    return manchester_syntax_query


##### NOT USING: query correcter, we now use one of the helper functions loaded in section 2

In [36]:
def correct_query(question: str, query: str, model="llama3.2") -> str:
    val_prompt: str = f"""
    Check the question and the query and ensure the
    query properly maps all the requirements from the question.

    Question: {question}

    Query: {query}

    ### Vocabulary Reference
    - **Properties:** hasFacility, location, userRating, nationality, restaurantType, inCity, diet, nextTo, inCountry, walkingDistance.
    - **Classes:** Restaurant, Hotel, Camping_Site, Hostel, Neighbourhood, Museum, City, TrainStation, PublicTransport, BodyOfWater, TouristAttraction, RestaurantType, UserRating, Country, Diet, Accommodation, River, Gold River, Ocean, Lagoon, Drainage Basin, Bay, Gulf, Sea, Adjacent Sea, Mediterranean Sea, Watercourse, Canal, Main Stream.
    - **Facility Values:** Sauna, 24h_front_desk, Airport_Shuttle, Free_Wifi, Parking_Space, Private_Bathroom, RestaurantInHotel, Swimming_Pool.
    - **RestaurantType Values:** FastFood, Fusion, Michelin, StreetFood, Traditional.
    - **UserRating Values:** 1_stars, 2_stars, 3_stars, 4_stars, 5_stars.

    Examples:
    - Question: "Could you recommend some hotels in Lisbon that are within walking distance of tourist attractions."
    - Query (Incorrect): Hotel and (inCity value Lisbon) and (nextTo some TouristAttraction)
    - Query (Correct): Hotel and (inCity value Lisbon) and (walkingDistance some TouristAttraction)

    - Question: "Hostel that is a walking distance from a museum."
    - Query (Incorrect): Hostel (nextTo some Museum)
    - Query (Correct): Hostel (walkingDistance some Museum)

    Output only the corrected query string without any introductory text or explanations.
    """

    response = ollama.chat(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You are a query correction assistant. Output only the final query string.'},
            {'role': 'user', 'content': val_prompt}
        ],
        options={
            'temperature': 0
        }
    )

    return response['message']['content']

In [ ]:
def _correct_query(query: str, model="llama3.2") -> str:
    #     - **Properties:** hasFacility, location, userRating, nationality, restaurantType, inCity, diet, nextTo, inCountry.
    validation_prompt = f"""
    ### Task
    Review the following Manchester Syntax query. Fix it ONLY if it violates the some/value rules.

    ### Rules
    1. Use 'value' when the object is a specific individual (e.g., 5_stars, Paris, Vegan, Swimming_Pool,).
    2. Use 'some' when the object is a general Class (e.g., River, Museum, HighRating, BodyOfWater).

    ### Reference
    - **Individuals (Use 'value'):**
    UserRating: 1_stars, 2_stars, 3_stars, 4_stars, 5_stars.
    Facility Values: Sauna, 24h_front_desk, Airport_Shuttle, Free_Wifi, Parking_Space, Private_Bathroom, RestaurantInHotel, Swimming_Pool.
    RestaurantType: FastFood, Fusion, Michelin, StreetFood, Traditional.
    - **Classes (Use 'some'):** Restaurant, Hotel, Camping_Site, Hostel, Neighbourhood, Museum, City, TrainStation, PublicTransport, BodyOfWater, TouristAttraction, RestaurantType, UserRating, Country, Diet, Accommodation, River,
    Gold River, Ocean, Lagoon, Drainage Basin, Bay, Gulf, Sea, Adjacent Sea, Mediterranean Sea,
    Watercourse, Canal, Main Stream.

    ### Input Query
    {query}

    ### Output Requirement
    - Output ONLY the corrected query string.
    - If it is already correct, output the original string.
    - No explanations or punctuation.

    ### Example
    - Input query: Hotel and (nextTo value Sea)
    - Corrected output query: Hotel and (nextTo some Sea)

    - Input query: Restaurant and (userRating some 5_stars)
    - Corrected output query: Restaurant and (userRating value 5_stars)
    """

    response = ollama.chat(
        model=model,
        messages=[
            {'role': 'user', 'content': validation_prompt}
        ],
        options={
            'temperature': 0
        }
    )

    return response['message']['content']

In [ ]:
parse_and_fix_manchester("Camping_Site and (nextTo value River) and (userRating some 5_stars)") #test case

'Camping_Site and (nextTo some River) and (userRating value 5_stars)'

##### Run below to generate the 10 dialogue-query pairs

In [ ]:
#Step 3: Run the LLM for each example defined above

# Helper function
def find_between(s: str, start: str, end: str) -> str:
    return s.split(start)[1].split(end)[0]

queries=[]

for dialogue in dialogues:
    print("User question:", dialogue)
    print()
    result: str = question_to_query(instruction, dialogue)
    # Possibly only extract the relevant parts
    print("Extracted query:", result)
    queries.append(result)
    print()
    print(50*"-")

User question: Find me an accomodation in Portugal which is near a landmark.

Extracted query: Accommodation and (near some Landmark) in Portugal

User question: I want to go to a camping which has free Wifi and near public transport

Extracted query: Camping_Site and (hasFacility value Free_Wifi) and (nextTo some PublicTransport)

User question: What are restaurantrs that serve vegan food.

Extracted query: Restaurant and (diet value Vegan)

User question: Find me all accomodations rated 5 stars.

Extracted query: Accommodation and (userRating some 5_stars)

User question: Restaurants next to a river.

Extracted query: Restaurant and (nextTo some BodyOfWater)

User question: I want a hotel with a sauna and a swimming pool, preferrably in Spain near the sea.

Extracted query: Hotel and (hasFacility value Sauna) and (hasFacility value Swimming_Pool) and (inCountry value Spain) and (nextTo some Sea).

User question: Find me a hotel that is also a michelin restaurant in France. They shoul

# 5. Use an LLM to summarize some result into a natural language response to the user.

In [143]:
#Step 1: Extract knowledge from query with the reasoner and return as list

def reason(query: str, question:str="", namespace: str = "http://kai.cs.vu.nl/2024/situated-minor-project/hotel#") -> tuple[list[str], str]:
    '''
    This function should convert a query into an OWL expression and use the reasoner
    to return the answers.
    Uses OWLAPY structural reasoner initialized earlier. Converts Manchester Syntax Query into OWL
    class expression.
    Converts generator object to list.
    '''
    # first replace wdt names by their uri
    query_resolved = resolve_manchester_fillers(query)
    # then fix some/ value errors
    query_fixed = parse_and_fix_manchester(query_resolved)
    # then convert this to a valid owl expression
    owl_class_expression = parse_manchester_query(query_fixed) # convert manch query to OWL class expression
    print("Corrected and resolved query:", query_resolved)
    owl_class_expression = parse_manchester_query(query_resolved) # convert manch query to OWL class expression
    # print("Query converted to OWL ce: \n",owl_class_expression)

    instances = []

    if pellet_reasoner.is_satisfiable(owl_class_expression):
        instances = list(pellet_reasoner.instances(owl_class_expression))
        if instances:
            return [i.iri._remainder if not i.iri.as_str()[:15] == "http://www.wiki" else get_label_or_remainder(i.iri) for i in instances], query_fixed
    else:
        print("First class expression is not satisfiable.")

    # second attempt with second llm checker for the query for if properties did not get appropriately linked to user question.

    second_query = correct_query(question, query)
    second_query_resolved = resolve_manchester_fillers(second_query)
    second_query_fixed = parse_and_fix_manchester(second_query_resolved)
    second_owl_class_expression = parse_manchester_query(second_query_fixed)
    print("Second corrected and resolved query:", second_query_fixed)
    # print("Query converted to OWL ce: \n",owl_class_expression)

    if pellet_reasoner.is_satisfiable(owl_class_expression):
        instances = list(pellet_reasoner.instances(owl_class_expression))
        if instances:
            return [i.iri._remainder if not i.iri.as_str()[:15] == "http://www.wiki" else get_label_or_remainder(i.iri) for i in instances], query_fixed
    else:
        print("Second class expression is not satisfiable.")

    return instances, "" # return default (empty results list and empty string) Tuple

def get_label_or_remainder(iri: str) -> IRI | str:
    """
    Takes as input some iri str object, outputs IRI remainder if it has no rdfs:label for representation
    useful if queries might return wdt instances
    """

    entity = default_world[iri]
    if entity:
        if hasattr(entity, "label") and entity.label:
            return entity.label[0]
        return entity
    return iri._remainder

In [136]:
get_label_or_remainder("http://www.wikidata.org/entity/Q36433")

'Porto'

<!-- ##### Test case: complex query -->

In [24]:
# USEFUL TO RUN WHEN IN COLAB IF OLLAMA cannot connect

# import subprocess

# import time

subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull llama3.2

In [13]:
onto = get_ontology(owl_file_path).load() #from owlready2, load once for get_label function

In [156]:
#Step 2: Instruct & run the LLM for the new task: transform the extracted knowledge into a natural language response based
# on the original question

def knowledge_to_response(question: str, query: str, knowledge: list[str], model="llama3.2"):
    """
    Transforms KG query results into a structured natural language response
    while strictly preventing hallucinations.
    """

    prompt: str = f"""
    ### ROLE
    You are a strict Data Reporter. Your task is to report results from a Knowledge Graph query.

    ### STRICT RULES
    1. USE ONLY the provided knowledge in **Retrieved matching individuals:**.
    2. DO NOT use external knowledge. If an instance is 'Museum_of_Art', do not describe its collection unless that description is in the retrieved knowledge.
    3. NO HALLUCINATIONS: If no information is provided about an instance other than its name, simply list its name.
    4. If 'Retrieved matching individuals' is empty or "NONE", the Results section must say "No matches found."

    ### INPUT DATA
    - **Original User Question:** {question}
    - **Logic Query:** {query}
    - **Retrieved matching individuals:** {knowledge}

    ### REQUIRED OUTPUT FORMAT
    **Specified Criteria:**
    * [List criteria here]

    **Results (list and include each item as a bullet point entry from the retrieved matching individuals):**
    * [List individual labels or IRIs]

    ---
    **Summary:**
    [A 1-sentence statement confirming how many matches were found for the criteria.]
    """

    response = ollama.chat(
      model=model,
      messages=[
          {'role': 'user', 'content': prompt},
      ],
      options={
          'temperature': 0
      }
    )

    return response['message']['content']


In [146]:
#Step 3: Combine everything: generate queries from the dialogues, extract knowledge from queries with the reasoner and
# generate summary responses

def run_pipeline(questions:list[str]) -> list[str]:
    """
    runs the whole pipeline:
    1) question to query
    2) query to correct manchester query
    3) correct manch query to OWL expression
    4) OWL expression for retrieving instances in local KG
    5) returning a list of the instances (their labels or remainder)
    6) retrieved knowledge used in answer with llm summarization prompt
    7) append
    """
    everything_list_of_dict = [
    ]
    for ind, question in enumerate(questions):
        print(30*"=")
        print(f"Question {ind+1}:", question)
        query = question_to_query(instruction, question)
        print("Ollama generated query:", query)
        knowl, final_query = reason(query, question)
        print("Individuals extracted from knowledge graph:\n", knowl)
        result = knowledge_to_response(
            question,
            final_query,
            knowl
        )

        print(f"Ollama summarized response:\n{result}")
        print(30*"=")
        print()
        everything_list_of_dict.append(
            {
                "dialogue": question,
                "initial query": query,
                "final query": final_query,
                "knowledge": knowl,
                "summary response": result
            }
        )
    return everything_list_of_dict

In [153]:
ke_responses = run_pipeline(dialogues)

Question 1: Find me an accomodation that is in Portugal and near a landmark.
Ollama generated query: Accommodation and (inCountry value Portugal) and (nextTo some Landmark)
Corrected and resolved query: Accommodation and (inCountry value http://www.wikidata.org/entity/Q45) and (nextTo some Landmark)
Second corrected and resolved query: Accommodation and (inCountry value http://www.wikidata.org/entity/Q45) and (nextTo some Landmark)
Individuals extracted from knowledge graph:
 []
Ollama summarized response:
**Specified Criteria:**

* Accommodation
* In Portugal
* Near a landmark

**Results:**

No matches found.

**Summary:** No matches found for the specified criteria of accommodation in Portugal near a landmark.

Question 2: I want to go to a camping by a river which is rated 5_stars.
Ollama generated query: Camping_Site and (nextTo some River) and (userRating some 5_stars)
Corrected and resolved query: Camping_Site and (nextTo some http://www.wikidata.org/entity/Q4022) and (userRating

In [155]:
# save generated responses to json
import json
if not colab:
    path = 'data_results5.json'
else:
    path = '/content/drive/MyDrive/psai/data_results5.json'
with open(path, 'w', encoding='utf-8') as f:
    json.dump(ke_responses, f, ensure_ascii=False, indent=4)

# 6. Evaluate your LLM

In [ ]:
# TODO: your code to implement and demonstrate evaluation metrics
# Suggestions: comparison of generated queries with the queries manually created in examples.txt, Intersection Over Union,
# but you can be creative here